# This is the Total Facricanova Backend System (Final Still Now and it is memory optimized) and it is related with the colab also

In [1]:
import torch
import gc
from diffusers import (
    StableDiffusionXLAdapterPipeline,
    T2IAdapter,
    EulerDiscreteScheduler,
    AutoencoderKL,
    UNet2DConditionModel
)
from transformers import CLIPVisionModelWithProjection
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# --- MEMORY CLEANUP ---
gc.collect()
torch.cuda.empty_cache()

print("🚀 Loading Fabricanova AI Models (FP16 Lightweight Mode)...")

# --- A. Load Image Encoder manually in FP16 (Saves ~2GB RAM) ---
# This prevents the pipeline from trying to load the massive default version later
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter",
    subfolder="sdxl_models/image_encoder",
    torch_dtype=torch.float16
)

# --- B. Load T2I-Adapter (Sketch Control) ---
adapter = T2IAdapter.from_pretrained(
    "TencentARC/t2i-adapter-sketch-sdxl-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
)

# --- C. Load VAE (Fixed FP16) ---
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# --- D. Load UNet (Lightning) ---
base_id = "stabilityai/stable-diffusion-xl-base-1.0"
unet = UNet2DConditionModel.from_config(base_id, subfolder="unet").to("cpu", torch.float16)
unet.load_state_dict(load_file(hf_hub_download("ByteDance/SDXL-Lightning", "sdxl_lightning_4step_unet.safetensors"), device="cpu"))

# --- E. Assemble Pipeline ---
pipe = StableDiffusionXLAdapterPipeline.from_pretrained(
    base_id,
    unet=unet,
    vae=vae,
    adapter=adapter,
    image_encoder=image_encoder, # Pass our lightweight encoder here
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)

# --- F. Critical Memory Guards ---
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

# Load the IP-Adapter weights (Fabric Texture)
# Since we already passed 'image_encoder', this won't trigger the memory spike
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")

# Aggressive offloading (Essential for Colab T4)
pipe.enable_sequential_cpu_offload()
pipe.enable_vae_tiling()
pipe.enable_vae_slicing()

print("✅ Model Ready! You can now run the Backend Server code.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!


🚀 Loading Fabricanova AI Models (FP16 Lightweight Mode)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

sdxl_models/image_encoder/model.safetens(…):   0%|          | 0.00/3.69G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

... (20 outputs left out)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

sdxl_models/ip-adapter_sdxl.bin:   0%|          | 0.00/703M [00:00<?, ?B/s]

✅ Model Ready! You can now run the Backend Server code.


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(
/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLAdapterPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [2]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install -q flask pyngrok rembg[gpu] onnxruntime-gpu flask-cors

import os
import io
import cv2
import base64
import numpy as np
import torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from PIL import Image, ImageDraw, ImageFont
from rembg import remove

# ==========================================
# 2. HELPER FUNCTIONS (Image Processing)
# ==========================================

def image_to_base64(img):
    """Converts a PIL Image to Base64 string for API response"""
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

def process_sketch(sketch_file):
    """Converts uploaded sketch to Canny Edge Map (White lines on Black)"""
    sketch = Image.open(sketch_file).convert("RGB").resize((1024, 1024))
    img_array = np.array(sketch)

    # Standard Canny Edge Detection
    canny_edges = cv2.Canny(img_array, 100, 200)

    # Dilation to make lines visible to AI
    dilated_edges = cv2.dilate(canny_edges, np.ones((2,2), np.uint8), iterations=1)

    return Image.fromarray(dilated_edges).convert("RGB")

def process_swatch(swatch_file):
    """Crops fabric swatch to prevent 'wallpaper' effect"""
    swatch = Image.open(swatch_file).convert("RGB").resize((1024, 1024))
    # Crop center 80%
    return swatch.crop((100, 100, 924, 924))

def generate_palette(image, k=5):
    """Extracts 5 dominant colors and returns Hex codes"""
    # Quantize image to k colors
    quantized = image.quantize(colors=k, method=2)
    palette = quantized.getpalette()[:k*3]

    # Convert RGB to Hex
    hex_codes = ['#{:02x}{:02x}{:02x}'.format(*tuple(palette[i:i+3])) for i in range(0, len(palette), 3)]
    return hex_codes

def create_lookbook(sketch, swatch, result, hex_codes):
    """Generates a professional presentation board"""
    # Create white canvas (16:9 Aspect Ratio)
    width, height = 1600, 900
    board = Image.new("RGB", (width, height), (255, 255, 255))
    draw = ImageDraw.Draw(board)

    # --- Layout ---

    # 1. Sketch (Left Top)
    if sketch:
        sketch_res = sketch.resize((400, 533)) # 3:4 aspect
        board.paste(sketch_res, (50, 50))
        draw.text((50, 600), "ORIGINAL SKETCH", fill="gray")
    else:
        # Placeholder if no sketch
        draw.rectangle([(50, 50), (450, 583)], outline="#e0e0e0", width=2)
        draw.text((180, 300), "NO SKETCH", fill="#e0e0e0")

    # 2. Fabric Swatch (Left Bottom)
    if swatch:
        swatch_res = swatch.resize((250, 250))
        board.paste(swatch_res, (500, 50))
        draw.text((500, 310), "MATERIAL", fill="gray")

    # 3. Final Design (Center/Right - Hero Image)
    # Resize to fit height while maintaining aspect ratio
    res_h = 800
    res_w = int(res_h * result.width / result.height)
    res_img = result.resize((res_w, res_h))
    board.paste(res_img, (800, 50))

    # 4. Color Palette (Vertical Bar on Right)
    x_start = 1450
    y_start = 50
    draw.text((x_start, 30), "PALETTE", fill="gray")

    for hex_c in hex_codes:
        # Parse Hex to RGB
        color = tuple(int(hex_c.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
        # Draw Swatch
        draw.rectangle([(x_start, y_start), (x_start+100, y_start+100)], fill=color)
        # Draw Label
        draw.text((x_start, y_start+110), hex_c, fill="black")
        y_start += 150

    return board

# ==========================================
# 3. FLASK API
# ==========================================

app = Flask(__name__)
CORS(app) # Enable Cross-Origin Resource Sharing

@app.route('/generate', methods=['POST'])
def generate():
    try:
        print("⚡ Request Received...")

        # --- A. SETUP INPUTS ---
        final_sketch = None
        sketch_scale = 0.0
        cropped_swatch = None
        ip_scale = 0.0

        # 1. Handle Prompt
        user_prompt = request.form.get('prompt', "")
        if not user_prompt:
            user_prompt = "A high fashion garment, studio lighting, highly detailed, 8k"

        # 2. Handle Sketch
        if 'sketch' in request.files and request.files['sketch'].filename != '':
            print("   -> Processing Sketch")
            final_sketch = process_sketch(request.files['sketch'])
            sketch_scale = 0.7
        else:
            # Dummy sketch (ignored by pipeline)
            final_sketch = Image.new("RGB", (1024, 1024), (0, 0, 0))

        # 3. Handle Fabric
        if 'swatch' in request.files and request.files['swatch'].filename != '':
            print("   -> Processing Fabric")
            cropped_swatch = process_swatch(request.files['swatch'])
            ip_scale = 0.6
        else:
            # Dummy swatch (ignored by pipeline)
            cropped_swatch = Image.new("RGB", (1024, 1024), (255, 255, 255))

        # --- B. GENERATION PIPELINE ---
        print(f"   -> Generating with Scales: Sketch={sketch_scale}, Fabric={ip_scale}")

        # Configure Dynamic Adapters
        pipe.set_ip_adapter_scale(ip_scale)

        raw_output = pipe(
            prompt=f"{user_prompt}, white background, 8k",
            negative_prompt="cartoon, low quality, messy, deformed, text, watermark, dark background, forest, trees",

            # Adapters
            image=final_sketch,
            adapter_conditioning_scale=sketch_scale,
            ip_adapter_image=cropped_swatch,

            num_inference_steps=4,
            guidance_scale=0
        ).images[0]

        # --- C. POST-PROCESSING ---
        print("   -> Removing Background")
        # 1. Remove Background
        clean_output = remove(raw_output)

        # 2. Composite onto White
        final_bg = Image.new("RGB", clean_output.size, (255, 255, 255))
        final_bg.paste(clean_output, mask=clean_output.split()[3])

        # 3. Extract Palette
        print("   -> Extracting Palette")
        hex_codes = generate_palette(final_bg)

        # 4. Create Lookbook
        print("   -> Assembling Lookbook")
        # Re-open original files for the lookbook (if they exist)
        sketch_input = Image.open(request.files['sketch']) if sketch_scale > 0 else None
        swatch_input = Image.open(request.files['swatch']) if ip_scale > 0 else None

        lookbook_img = create_lookbook(sketch_input, swatch_input, final_bg, hex_codes)

        # --- D. RESPONSE ---
        return jsonify({
            "status": "success",
            "image": image_to_base64(final_bg),       # The Main Design
            "lookbook": image_to_base64(lookbook_img), # The Presentation Slide
            "palette": hex_codes                      # The Color Codes
        })

    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return jsonify({"error": str(e)}), 500

# ==========================================
# 4. START SERVER
# ==========================================

# REPLACE WITH YOUR ACTUAL TOKEN
ngrok.set_auth_token("2vM0B4m36n5BdVFgIXqOPmS1Rsb_47BYD7LzfGdBGttGXTwY6")

# Connect to Ngrok
public_url = ngrok.connect(5000)
print(f"🚀 Fabricanova Backend is LIVE at: {public_url}")

# Run Flask App
app.run(port=5000)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 142.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [14/Jan/2026 13:19:08] "OPTIONS /generate HTTP/1.1" 200 -


⚡ Request Received...
   -> Processing Sketch
   -> Generating with Scales: Sketch=0.7, Fabric=0.0


  0%|          | 0/4 [00:00<?, ?it/s]

   -> Removing Background


100%|████████████████████████████████████████| 176M/176M [00:00<00:00, 268GB/s]


... (26 outputs left out)

⚡ Request Received...
   -> Processing Sketch
   -> Processing Fabric
   -> Generating with Scales: Sketch=0.7, Fabric=0.6


  0%|          | 0/4 [00:00<?, ?it/s]

   -> Removing Background
   -> Extracting Palette
   -> Assembling Lookbook


INFO:werkzeug:127.0.0.1 - - [14/Jan/2026 13:46:57] "POST /generate HTTP/1.1" 200 -
